# Data Setup, EDA, dan Split

Notebook ini menyiapkan konfigurasi, memuat dataset, menjalankan EDA pada gambar asli, lalu membuat split data di cell paling akhir.

# Face Mask Detection: Full CV Pipeline From Scratch

Notebook ini membangun pipeline computer vision lengkap untuk klasifikasi penggunaan masker wajah. Seluruh model deep learning dibuat manual menggunakan layer dasar `Conv2D` dan dilatih dari awal tanpa bobot eksternal.

## 1. Desain Eksperimen

Dataset yang digunakan adalah Face Mask Dataset dari Kaggle dengan dua kelas utama: `with_mask` dan `without_mask`. Split dibuat sekali di level file asli dengan rasio 70/15/15 untuk train, validation, dan test. Semua skenario memakai split yang sama agar tidak terjadi data leakage.

Pipeline akademik yang diuji:

- preprocessing: resize 128x128, normalisasi 0-1, label binary
- enhancement/restoration: CLAHE dan Gaussian denoise ringan
- face segmentation/cropping: Haar Cascade dengan fallback ke full image
- feature extraction klasik: HOG
- classification klasik: SVM
- classification deep learning: custom CNN Conv2D from scratch dengan opsi SE block
- evaluasi: accuracy, precision, recall, F1-score, classification report, confusion matrix, ROC curve, AUC, dan kurva training

Skenario wajib:

- HOG+SVM vs CNN
- enhancement vs tanpa enhancement
- crop vs full image
- augmentation vs tanpa augmentation
- SE block vs tanpa SE block

Catatan penting: notebook ini tidak memakai model siap pakai, metode pemindahan bobot, atau bobot eksternal. Arsitektur CNN dibuat manual dari layer `Conv2D`.

## 2. Setup

Konfigurasi dibuat eksplisit agar eksperimen mudah direproduksi dan mudah dijelaskan saat presentasi/laporan.

In [ ]:
import json
import os
import random
import shutil
import subprocess
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    auc,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

SEED = 42
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
MAX_EPOCHS = 50
EARLY_STOP_PATIENCE = 8
LR_PATIENCE = 3

RUN_OPTIONAL_CROP_SCENARIO = True
RUN_SE_ABLATION = True
RUN_LIGHT_TUNING = True

CLASS_NAMES = ['with_mask', 'without_mask']
CLASS_TO_LABEL = {name: idx for idx, name in enumerate(CLASS_NAMES)}
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

OUTPUT_DIR = Path('/kaggle/working')
TEMP_DIR = Path('/kaggle/temp')
DEFAULT_DATASET_DIR = Path('/kaggle/input/face-mask-dataset')
SCENARIO_ROOT = TEMP_DIR / 'face_mask_scenarios_final'
DATASET_HANDLE = 'omkargurav/face-mask-dataset'

os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)

print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

TensorFlow: 2.20.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 3. Load Dataset Kaggle

Notebook mencari folder dataset yang berisi class `with_mask` dan `without_mask`. Jika dataset belum ter-mount di Kaggle, fallback download disediakan untuk menjaga notebook tetap reproducible.

In [ ]:
def print_tree(root: Path, max_depth: int = 3, limit: int = 120) -> None:
    print(f'Dataset tree preview: {root}')
    if not root.exists():
        print('Dataset path does not exist.')
        return

    root_depth = len(root.parts)
    shown = 0
    for path in sorted(root.rglob('*')):
        depth = len(path.parts) - root_depth
        if depth > max_depth:
            continue
        indent = '  ' * max(depth - 1, 0)
        suffix = '/' if path.is_dir() else ''
        print(f'{indent}{path.name}{suffix}')
        shown += 1
        if shown >= limit:
            print('... tree preview truncated ...')
            break


def find_image_directory(dataset_root: Path) -> Path:
    directories = [dataset_root]
    if dataset_root.exists():
        directories.extend([p for p in dataset_root.rglob('*') if p.is_dir()])

    for directory in directories:
        child_dirs = {p.name.lower(): p for p in directory.iterdir() if p.is_dir()}
        if all(name in child_dirs for name in CLASS_NAMES):
            return directory

    raise FileNotFoundError(f'Could not find class folders {CLASS_NAMES} under {dataset_root}')


def find_mounted_dataset() -> Path | None:
    if DEFAULT_DATASET_DIR.exists():
        return DEFAULT_DATASET_DIR

    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        print('Available /kaggle/input entries:', [p.name for p in kaggle_input.iterdir()])
        for pattern in ['*face*mask*', '*mask*', '*face*']:
            matches = [p for p in kaggle_input.glob(pattern) if p.is_dir()]
            if matches:
                return matches[0]
    return None


def download_dataset_fallback() -> Path:
    download_dir = OUTPUT_DIR / 'downloaded_face_mask_dataset'
    download_dir.mkdir(parents=True, exist_ok=True)

    try:
        import kagglehub

        print('Dataset mount not found. Trying kagglehub fallback download...')
        downloaded_path = Path(kagglehub.dataset_download(DATASET_HANDLE))
        print('kagglehub downloaded dataset to:', downloaded_path)
        return downloaded_path
    except Exception as error:
        print('kagglehub fallback failed:', repr(error))

    print('Trying Kaggle CLI fallback download...')
    subprocess.run(
        ['kaggle', 'datasets', 'download', '-d', DATASET_HANDLE, '-p', str(download_dir), '--unzip'],
        check=True,
    )
    return download_dir


dataset_root = find_mounted_dataset()
if dataset_root is None:
    dataset_root = download_dataset_fallback()

print_tree(dataset_root)
original_image_dir = find_image_directory(dataset_root)
print('Selected image directory:', original_image_dir)

Available /kaggle/input entries: []
Dataset mount not found. Trying kagglehub fallback download...


100%|██████████| 163M/163M [00:09<00:00, 17.9MB/s]

Extracting files...


kagglehub downloaded dataset to: /root/.cache/kagglehub/datasets/omkargurav/face-mask-dataset/versions/1
Dataset tree preview: /root/.cache/kagglehub/datasets/omkargurav/face-mask-dataset/versions/1
data/
  with_mask/
    with_mask_1.jpg
    with_mask_10.jpg
    with_mask_100.jpg
    with_mask_1000.jpg
    with_mask_1001.jpg
    with_mask_1002.jpg
    with_mask_1003.jpg
    with_mask_1004.jpg
    with_mask_1005.jpg
    with_mask_1006.jpg
    with_mask_1007.jpg
    with_mask_1008.jpg
    with_mask_1009.jpg
    with_mask_101.jpg
    with_mask_1010.jpg
    with_mask_1011.jpg
    with_mask_1012.jpg
    with_mask_1013.jpg
    with_mask_1014.jpg
    with_mask_1015.jpg
    with_mask_1016.jpg
    with_mask_1017.jpg
    with_mask_1018.jpg
    with_mask_1019.jpg
    with_mask_102.jpg
    with_mask_1020.jpg
    with_mask_1021.jpg
    with_mask_1022.jpg
    with_mask_1023.jpg
    with_mask_1024.jpg
    with_mask_1025.jpg
    with_mask_1026.jpg
    with_mask_1027.jpg
    with_mask_1028.jpg
    with

## 4. Inventaris Dataset Asli

Path gambar dikumpulkan per class tanpa mengubah citra. Inventaris ini menjadi dasar semua analisis sebelum preprocessing dan split data.

In [ ]:

def collect_image_paths(image_dir: Path) -> dict[str, list[Path]]:
    paths_by_class = {}
    for class_name in CLASS_NAMES:
        class_dir = image_dir / class_name
        if not class_dir.exists():
            raise FileNotFoundError(f'Missing class folder: {class_dir}')
        paths = [p for p in sorted(class_dir.rglob('*')) if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS]
        if not paths:
            raise ValueError(f'No images found for class: {class_name}')
        paths_by_class[class_name] = paths
    return paths_by_class


def load_original_rgb(image_path: Path) -> np.ndarray | None:
    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        return None
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)


paths_by_class = collect_image_paths(original_image_dir)
class_counts = {class_name: len(paths) for class_name, paths in paths_by_class.items()}
print('Class counts:', class_counts)
print('Total images:', sum(class_counts.values()))


## 5. Distribusi Kelas

Jumlah gambar pada kelas `with_mask` dan `without_mask` dihitung lalu divisualisasikan dalam bar chart. Analisis ini bertujuan melihat keseimbangan data sejak awal, karena komposisi kelas akan memengaruhi cara membaca performa akhir seperti accuracy, precision, recall, F1-score, dan AUC.


In [ ]:

class_values = [class_counts[class_name] for class_name in CLASS_NAMES]

plt.figure(figsize=(7, 4))
bars = plt.bar(CLASS_NAMES, class_values, color=['#2f80ed', '#27ae60'])
plt.title('Distribusi Class Dataset Asli')
plt.xlabel('Class')
plt.ylabel('Jumlah gambar')
plt.bar_label(bars)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_class_distribution.png', dpi=160)
plt.show()


## 6. Visualisasi Sampel Gambar

Beberapa gambar asli dari setiap kelas ditampilkan secara acak tanpa preprocessing. Alurnya dimulai dari mengambil sampel per kelas, menampilkan citra dalam grid, lalu mengamati kesesuaian label, variasi bentuk masker, ekspresi wajah, jarak kamera, pencahayaan, dan kondisi background.


In [ ]:

rng = random.Random(SEED)
fig, axes = plt.subplots(len(CLASS_NAMES), 6, figsize=(13, 5))
for row_idx, class_name in enumerate(CLASS_NAMES):
    sample_paths = rng.sample(paths_by_class[class_name], k=min(6, len(paths_by_class[class_name])))
    for col_idx, image_path in enumerate(sample_paths):
        image_rgb = load_original_rgb(image_path)
        axes[row_idx, col_idx].imshow(image_rgb)
        axes[row_idx, col_idx].set_title(class_name, fontsize=9)
        axes[row_idx, col_idx].axis('off')
plt.suptitle('Contoh Gambar Asli Sebelum Preprocessing', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_original_samples.png', dpi=160, bbox_inches='tight')
plt.show()


## 7. Distribusi Resolusi Gambar

Lebar, tinggi, dan rasio aspek setiap gambar asli dihitung sebelum ukuran gambar diseragamkan. Analisis ini bertujuan memahami variasi dimensi dataset, melihat apakah ada gambar yang terlalu kecil atau sangat tidak proporsional, dan menentukan konsekuensi resize terhadap detail wajah serta masker.


In [ ]:
def collect_eda_image_stats(paths_by_class: dict[str, list[Path]]) -> tuple[list[dict], list[str]]:
    stats = []
    read_failed = []
    for class_name in CLASS_NAMES:
        for image_path in paths_by_class[class_name]:
            image_rgb = load_original_rgb(image_path)
            if image_rgb is None:
                read_failed.append(str(image_path))
                continue
            gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
            height, width = gray.shape
            stats.append(
                {
                    'path': str(image_path),
                    'class_name': class_name,
                    'width': int(width),
                    'height': int(height),
                    'aspect_ratio': float(width / height),
                    'brightness_mean': float(gray.mean()),
                    'contrast_std': float(gray.std()),
                }
            )
    return stats, read_failed


eda_image_stats, eda_read_failed = collect_eda_image_stats(paths_by_class)
print('EDA image count:', len(eda_image_stats))
print('EDA read failed:', len(eda_read_failed))

widths = np.array([item['width'] for item in eda_image_stats])
heights = np.array([item['height'] for item in eda_image_stats])
aspect_ratios = np.array([item['aspect_ratio'] for item in eda_image_stats])
class_labels_for_stats = np.array([item['class_name'] for item in eda_image_stats])

colors = ['#2f80ed', '#f2994a']
fig, axes = plt.subplots(2, len(CLASS_NAMES), figsize=(6 * len(CLASS_NAMES), 7), squeeze=False)

for col_idx, (class_name, color) in enumerate(zip(CLASS_NAMES, colors)):
    class_mask = class_labels_for_stats == class_name
    class_widths = widths[class_mask]
    class_heights = heights[class_mask]
    class_aspect_ratios = aspect_ratios[class_mask]

    axes[0, col_idx].scatter(class_widths, class_heights, alpha=0.25, s=12, color=color)
    axes[0, col_idx].axvline(IMAGE_SIZE[0], color='black', linestyle='--', linewidth=1, label='target width')
    axes[0, col_idx].axhline(IMAGE_SIZE[1], color='gray', linestyle='--', linewidth=1, label='target height')
    axes[0, col_idx].set_title(f'Resolusi - {class_name}')
    axes[0, col_idx].set_xlabel('Width')
    axes[0, col_idx].set_ylabel('Height')
    axes[0, col_idx].legend()

    axes[1, col_idx].hist(class_aspect_ratios, bins=30, color=color, alpha=0.85, edgecolor='white')
    axes[1, col_idx].axvline(1.0, color='black', linestyle='--', linewidth=1, label='rasio 1:1')
    axes[1, col_idx].axvline(np.median(class_aspect_ratios), color=color, linestyle=':', linewidth=1.8, label='median')
    axes[1, col_idx].set_title(f'Aspect Ratio - {class_name}')
    axes[1, col_idx].set_xlabel('Width / Height')
    axes[1, col_idx].set_ylabel('Jumlah gambar')
    axes[1, col_idx].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_resolution_aspect_ratio.png', dpi=160)
plt.show()

eda_summary = {
    'image_count': int(len(eda_image_stats)),
    'read_failed_count': int(len(eda_read_failed)),
    'class_counts': class_counts,
    'width': {'min': int(widths.min()), 'median': float(np.median(widths)), 'max': int(widths.max())},
    'height': {'min': int(heights.min()), 'median': float(np.median(heights)), 'max': int(heights.max())},
    'aspect_ratio': {'min': float(aspect_ratios.min()), 'median': float(np.median(aspect_ratios)), 'max': float(aspect_ratios.max())},
    'figures': {
        'class_distribution': str(OUTPUT_DIR / 'eda_class_distribution.png'),
        'original_samples': str(OUTPUT_DIR / 'eda_original_samples.png'),
        'resolution_aspect_ratio': str(OUTPUT_DIR / 'eda_resolution_aspect_ratio.png'),
    },
}


## 8. Analisis Brightness dan Kontras

Setiap gambar dikonversi ke grayscale, lalu dihitung rata-rata intensitas sebagai brightness dan standar deviasi intensitas sebagai kontras. Distribusi nilai per kelas digunakan untuk membaca variasi pencahayaan dan ketajaman intensitas lokal, terutama apakah dataset banyak berisi gambar terlalu gelap, terlalu terang, atau rendah kontras.


In [ ]:
brightness = np.array([item['brightness_mean'] for item in eda_image_stats])
contrast = np.array([item['contrast_std'] for item in eda_image_stats])
brightness_by_class = [brightness[class_labels_for_stats == class_name] for class_name in CLASS_NAMES]
contrast_by_class = [contrast[class_labels_for_stats == class_name] for class_name in CLASS_NAMES]

colors = ['#2f80ed', '#f2994a']
fig, axes = plt.subplots(len(CLASS_NAMES), 2, figsize=(12, 4 * len(CLASS_NAMES)), squeeze=False)

for row_idx, (class_name, color, brightness_values, contrast_values) in enumerate(
    zip(CLASS_NAMES, colors, brightness_by_class, contrast_by_class)
):
    axes[row_idx, 0].hist(brightness_values, bins=30, alpha=0.85, color=color, edgecolor='white')
    axes[row_idx, 0].axvline(np.median(brightness_values), color='black', linestyle='--', linewidth=1.6, label='median')
    axes[row_idx, 0].set_title(f'Brightness - {class_name}')
    axes[row_idx, 0].set_xlabel('Mean grayscale')
    axes[row_idx, 0].set_ylabel('Jumlah gambar')
    axes[row_idx, 0].legend()

    axes[row_idx, 1].hist(contrast_values, bins=30, alpha=0.85, color=color, edgecolor='white')
    axes[row_idx, 1].axvline(np.median(contrast_values), color='black', linestyle='--', linewidth=1.6, label='median')
    axes[row_idx, 1].set_title(f'Kontras - {class_name}')
    axes[row_idx, 1].set_xlabel('Std grayscale')
    axes[row_idx, 1].set_ylabel('Jumlah gambar')
    axes[row_idx, 1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_brightness_contrast.png', dpi=160)
plt.show()

eda_summary['brightness_mean'] = {'min': float(brightness.min()), 'median': float(np.median(brightness)), 'max': float(brightness.max())}
eda_summary['contrast_std'] = {'min': float(contrast.min()), 'median': float(np.median(contrast)), 'max': float(contrast.max())}
eda_summary['figures']['brightness_contrast'] = str(OUTPUT_DIR / 'eda_brightness_contrast.png')


## 9. Analisis Noise dan Blur

Blur dihitung menggunakan variance of Laplacian, sedangkan noise diperkirakan dari standar deviasi residual antara gambar grayscale dan hasil Gaussian blur ringan. Analisis ini bertujuan melihat kualitas citra asli, membedakan gambar yang detailnya tajam dari gambar yang kabur, dan mengetahui seberapa besar gangguan tekstur halus pada dataset.


In [ ]:
def estimate_noise_blur(paths_by_class: dict[str, list[Path]]) -> list[dict]:
    records = []
    for class_name in CLASS_NAMES:
        for image_path in paths_by_class[class_name]:
            image_rgb = load_original_rgb(image_path)
            if image_rgb is None:
                continue
            gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
            blur_score = float(cv2.Laplacian(gray, cv2.CV_64F).var())
            smooth = cv2.GaussianBlur(gray, (3, 3), sigmaX=0.5)
            noise_score = float(np.std(gray.astype(np.float32) - smooth.astype(np.float32)))
            records.append({'class_name': class_name, 'blur_score': blur_score, 'noise_score': noise_score})
    return records


noise_blur_records = estimate_noise_blur(paths_by_class)
blur_scores = np.array([item['blur_score'] for item in noise_blur_records])
noise_scores = np.array([item['noise_score'] for item in noise_blur_records])
noise_blur_labels = np.array([item['class_name'] for item in noise_blur_records])

blur_by_class = [blur_scores[noise_blur_labels == class_name] for class_name in CLASS_NAMES]
noise_by_class = [noise_scores[noise_blur_labels == class_name] for class_name in CLASS_NAMES]
blur_log_by_class = [np.log1p(values) for values in blur_by_class]

colors = ['#2f80ed', '#f2994a']
fig, axes = plt.subplots(len(CLASS_NAMES), 2, figsize=(12, 4 * len(CLASS_NAMES)), squeeze=False)

for row_idx, (class_name, color, blur_values, noise_values) in enumerate(
    zip(CLASS_NAMES, colors, blur_log_by_class, noise_by_class)
):
    axes[row_idx, 0].hist(blur_values, bins=30, alpha=0.85, color=color, edgecolor='white')
    axes[row_idx, 0].axvline(np.median(blur_values), color='black', linestyle='--', linewidth=1.6, label='median')
    axes[row_idx, 0].set_title(f'Blur - {class_name}')
    axes[row_idx, 0].set_xlabel('log1p variance of Laplacian')
    axes[row_idx, 0].set_ylabel('Jumlah gambar')
    axes[row_idx, 0].legend()

    axes[row_idx, 1].hist(noise_values, bins=30, alpha=0.85, color=color, edgecolor='white')
    axes[row_idx, 1].axvline(np.median(noise_values), color='black', linestyle='--', linewidth=1.6, label='median')
    axes[row_idx, 1].set_title(f'Noise - {class_name}')
    axes[row_idx, 1].set_xlabel('Std residual')
    axes[row_idx, 1].set_ylabel('Jumlah gambar')
    axes[row_idx, 1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_noise_blur.png', dpi=160)
plt.show()

eda_summary['blur_score'] = {'min': float(blur_scores.min()), 'median': float(np.median(blur_scores)), 'max': float(blur_scores.max())}
eda_summary['noise_score'] = {'min': float(noise_scores.min()), 'median': float(np.median(noise_scores)), 'max': float(noise_scores.max())}
eda_summary['figures']['noise_blur'] = str(OUTPUT_DIR / 'eda_noise_blur.png')


## 10. Visualisasi Edge dan HOG

Sampel gambar ditampilkan bersama Canny edge dan visualisasi orientasi gradien. Alur ini membantu mengamati struktur tepi yang muncul pada area wajah, masker, hidung, mulut, dan kontur kepala, sehingga pola bentuk yang dominan pada citra dapat terlihat sebelum data masuk ke tahap pemodelan.


In [ ]:

def draw_hog_orientation(image_rgb: np.ndarray, cell_size: int = 8, bins: int = 9) -> np.ndarray:
    resized = cv2.resize(image_rgb, IMAGE_SIZE, interpolation=cv2.INTER_AREA)
    gray = cv2.cvtColor(resized, cv2.COLOR_RGB2GRAY)
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=1)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=1)
    magnitude, angle = cv2.cartToPolar(gx, gy, angleInDegrees=True)
    angle = angle % 180

    canvas = np.zeros_like(gray, dtype=np.uint8)
    bin_width = 180 / bins
    for y in range(0, gray.shape[0], cell_size):
        for x in range(0, gray.shape[1], cell_size):
            mag_cell = magnitude[y:y + cell_size, x:x + cell_size]
            ang_cell = angle[y:y + cell_size, x:x + cell_size]
            if mag_cell.size == 0:
                continue
            hist = np.zeros(bins, dtype=np.float32)
            bin_ids = np.minimum((ang_cell / bin_width).astype(int), bins - 1)
            for bin_id in range(bins):
                hist[bin_id] = mag_cell[bin_ids == bin_id].sum()
            if hist.max() <= 0:
                continue
            dominant = int(hist.argmax())
            theta = np.deg2rad((dominant + 0.5) * bin_width)
            length = int((hist[dominant] / (hist.max() + 1e-6)) * cell_size * 0.45)
            cx = x + cell_size // 2
            cy = y + cell_size // 2
            dx = int(np.cos(theta) * length)
            dy = int(np.sin(theta) * length)
            cv2.line(canvas, (cx - dx, cy - dy), (cx + dx, cy + dy), 255, 1)
    return canvas


fig, axes = plt.subplots(len(CLASS_NAMES), 3, figsize=(9, 6))
rng = random.Random(SEED + 7)
for row_idx, class_name in enumerate(CLASS_NAMES):
    image_path = rng.choice(paths_by_class[class_name])
    image_rgb = load_original_rgb(image_path)
    resized = cv2.resize(image_rgb, IMAGE_SIZE, interpolation=cv2.INTER_AREA)
    gray = cv2.cvtColor(resized, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, threshold1=80, threshold2=160)
    hog_view = draw_hog_orientation(image_rgb)

    axes[row_idx, 0].imshow(resized)
    axes[row_idx, 0].set_title(f'Original - {class_name}', fontsize=9)
    axes[row_idx, 1].imshow(edges, cmap='gray')
    axes[row_idx, 1].set_title('Canny Edge', fontsize=9)
    axes[row_idx, 2].imshow(hog_view, cmap='gray')
    axes[row_idx, 2].set_title('HOG Orientation', fontsize=9)
    for col_idx in range(3):
        axes[row_idx, col_idx].axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_edge_hog_examples.png', dpi=160)
plt.show()

eda_summary['figures']['edge_hog_examples'] = str(OUTPUT_DIR / 'eda_edge_hog_examples.png')


## 11. Analisis Variasi Pose dan Background

Deteksi wajah digunakan untuk memperkirakan area wajah, posisi wajah terhadap pusat gambar, serta variasi warna dan brightness pada area tepi sebagai gambaran background. Analisis ini bertujuan membaca keragaman pose, skala wajah, framing, dan kondisi latar yang dapat membuat gambar antar sampel berbeda walaupun label kelasnya sama.


In [ ]:
cascade_path = Path(cv2.data.haarcascades) / 'haarcascade_frontalface_default.xml'
eda_face_detector = cv2.CascadeClassifier(str(cascade_path))
face_detection_records = []
face_examples_detected = []
face_examples_failed = []

for item in eda_image_stats:
    image_path = Path(item['path'])
    image_rgb = load_original_rgb(image_path)
    height, width = image_rgb.shape[:2]
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    faces = eda_face_detector.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(35, 35))
    detected = len(faces) > 0

    border = max(4, int(0.08 * min(height, width)))
    border_pixels = np.concatenate(
        [
            image_rgb[:border, :, :].reshape(-1, 3),
            image_rgb[-border:, :, :].reshape(-1, 3),
            image_rgb[:, :border, :].reshape(-1, 3),
            image_rgb[:, -border:, :].reshape(-1, 3),
        ],
        axis=0,
    )
    background_color_std = float(border_pixels.std(axis=0).mean())
    background_brightness = float(cv2.cvtColor(border_pixels.reshape(-1, 1, 3), cv2.COLOR_RGB2GRAY).mean())

    face_area_ratio = None
    face_center_offset = None
    if detected:
        x, y, w, h = max(faces, key=lambda box: box[2] * box[3])
        face_area_ratio = float((w * h) / (width * height))
        face_cx = x + w / 2
        face_cy = y + h / 2
        face_center_offset = float(np.sqrt(((face_cx / width) - 0.5) ** 2 + ((face_cy / height) - 0.5) ** 2))
        if len(face_examples_detected) < 4:
            preview = image_rgb.copy()
            cv2.rectangle(preview, (x, y), (x + w, y + h), (255, 60, 60), 3)
            face_examples_detected.append((preview, item['class_name']))
    elif len(face_examples_failed) < 4:
        face_examples_failed.append((image_rgb, item['class_name']))

    face_detection_records.append(
        {
            **item,
            'face_detected': bool(detected),
            'face_area_ratio': face_area_ratio,
            'face_center_offset': face_center_offset,
            'background_color_std': background_color_std,
            'background_brightness': background_brightness,
        }
    )

face_area_ratios = np.array([record['face_area_ratio'] for record in face_detection_records if record['face_area_ratio'] is not None])
face_center_offsets = np.array([record['face_center_offset'] for record in face_detection_records if record['face_center_offset'] is not None])
background_color_std = np.array([record['background_color_std'] for record in face_detection_records])
background_brightness = np.array([record['background_brightness'] for record in face_detection_records])

pose_metrics = [
    ('face_area_ratio', 'Skala Wajah Terdeteksi', 'Face area / image area', '#2f80ed'),
    ('face_center_offset', 'Offset Posisi Wajah dari Tengah', 'Normalized center offset', '#27ae60'),
    ('background_color_std', 'Variasi Warna Background', 'Mean RGB std pada border', '#f2994a'),
    ('background_brightness', 'Brightness Background', 'Mean grayscale pada border', '#8856a7'),
]

fig, axes = plt.subplots(len(pose_metrics), len(CLASS_NAMES), figsize=(6 * len(CLASS_NAMES), 3.5 * len(pose_metrics)), squeeze=False)

for row_idx, (metric_key, title, xlabel, color) in enumerate(pose_metrics):
    for col_idx, class_name in enumerate(CLASS_NAMES):
        values = np.array(
            [
                record[metric_key]
                for record in face_detection_records
                if record['class_name'] == class_name and record[metric_key] is not None
            ]
        )
        axes[row_idx, col_idx].hist(values, bins=30, color=color, alpha=0.85, edgecolor='white')
        if len(values):
            axes[row_idx, col_idx].axvline(np.median(values), color='black', linestyle='--', linewidth=1.4, label='median')
            axes[row_idx, col_idx].legend()
        axes[row_idx, col_idx].set_title(f'{title} - {class_name}')
        axes[row_idx, col_idx].set_xlabel(xlabel)
        axes[row_idx, col_idx].set_ylabel('Jumlah gambar')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_pose_background_variation.png', dpi=160)
plt.show()

eda_summary['pose_background'] = {
    'face_area_ratio_median': float(np.median(face_area_ratios)) if len(face_area_ratios) else None,
    'face_center_offset_median': float(np.median(face_center_offsets)) if len(face_center_offsets) else None,
    'background_color_std_median': float(np.median(background_color_std)),
    'background_brightness_median': float(np.median(background_brightness)),
}
eda_summary['figures']['pose_background_variation'] = str(OUTPUT_DIR / 'eda_pose_background_variation.png')


## 12. Coverage Face Detection Haar Cascade

Haar Cascade dijalankan pada gambar asli untuk menghitung berapa banyak wajah yang berhasil terdeteksi dan menampilkan contoh keberhasilan maupun kegagalannya. Analisis ini bertujuan mengetahui cakupan deteksi wajah pada dataset, termasuk kondisi yang rawan gagal seperti pose miring, wajah kecil, blur, pencahayaan ekstrem, atau background yang ramai.


In [ ]:

face_detection_summary = {}
for class_name in CLASS_NAMES:
    records = [record for record in face_detection_records if record['class_name'] == class_name]
    detected_count = sum(record['face_detected'] for record in records)
    face_detection_summary[class_name] = {
        'total': int(len(records)),
        'detected': int(detected_count),
        'not_detected': int(len(records) - detected_count),
        'detection_rate': float(detected_count / len(records)) if records else 0.0,
    }

all_detected_count = sum(record['face_detected'] for record in face_detection_records)
face_detection_summary['overall'] = {
    'total': int(len(face_detection_records)),
    'detected': int(all_detected_count),
    'not_detected': int(len(face_detection_records) - all_detected_count),
    'detection_rate': float(all_detected_count / len(face_detection_records)) if face_detection_records else 0.0,
}
print('Face detection summary:', json.dumps(face_detection_summary, indent=2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
detection_rates = [face_detection_summary[class_name]['detection_rate'] * 100 for class_name in CLASS_NAMES]
bars = axes[0].bar(CLASS_NAMES, detection_rates, color=['#2f80ed', '#27ae60'])
axes[0].set_ylim(0, 100)
axes[0].set_title('Coverage Haar Cascade per Class')
axes[0].set_ylabel('Detection rate (%)')
axes[0].bar_label(bars, fmt='%.1f%%')

overall_values = [face_detection_summary['overall']['detected'], face_detection_summary['overall']['not_detected']]
axes[1].pie(overall_values, labels=['Detected', 'Not detected'], autopct='%1.1f%%', colors=['#27ae60', '#d9534f'])
axes[1].set_title('Coverage Haar Cascade Keseluruhan')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_face_detection_coverage.png', dpi=160)
plt.show()

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for idx, (image_rgb, class_name) in enumerate(face_examples_detected[:4]):
    axes[0, idx].imshow(image_rgb)
    axes[0, idx].set_title(f'Detected - {class_name}', fontsize=9)
    axes[0, idx].axis('off')
for idx, (image_rgb, class_name) in enumerate(face_examples_failed[:4]):
    axes[1, idx].imshow(image_rgb)
    axes[1, idx].set_title(f'Not detected - {class_name}', fontsize=9)
    axes[1, idx].axis('off')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_face_detection_examples.png', dpi=160)
plt.show()

eda_summary['face_detection_summary'] = face_detection_summary
eda_summary['figures'].update(
    {
        'face_detection_coverage': str(OUTPUT_DIR / 'eda_face_detection_coverage.png'),
        'face_detection_examples': str(OUTPUT_DIR / 'eda_face_detection_examples.png'),
    }
)


## 13. Split 70/15/15 Tanpa Data Leakage

Setelah seluruh analisis gambar asli selesai, daftar file dibagi menjadi train, validation, dan test dengan rasio 70/15/15 secara stratified. Tujuannya memastikan satu file asli hanya masuk ke satu split, menjaga distribusi kelas tetap proporsional, dan membuat seluruh tahap berikutnya memakai pembagian data yang sama.


In [ ]:

def split_paths(paths_by_class: dict[str, list[Path]]) -> dict[str, list[tuple[Path, str]]]:
    rng = random.Random(SEED)
    splits = {'train': [], 'validation': [], 'test': []}

    for class_name, paths in paths_by_class.items():
        shuffled = paths.copy()
        rng.shuffle(shuffled)
        n_total = len(shuffled)
        n_train = int(n_total * TRAIN_RATIO)
        n_val = int(n_total * VAL_RATIO)

        splits['train'].extend((path, class_name) for path in shuffled[:n_train])
        splits['validation'].extend((path, class_name) for path in shuffled[n_train:n_train + n_val])
        splits['test'].extend((path, class_name) for path in shuffled[n_train + n_val:])

    for split_name in splits:
        rng.shuffle(splits[split_name])
    return splits


def count_split_labels(splits: dict[str, list[tuple[Path, str]]]) -> dict[str, dict[str, int]]:
    counts = {}
    for split_name, records in splits.items():
        counts[split_name] = {class_name: 0 for class_name in CLASS_NAMES}
        for _, class_name in records:
            counts[split_name][class_name] += 1
    return counts


splits = split_paths(paths_by_class)
split_counts = count_split_labels(splits)
split_sizes = {split_name: len(records) for split_name, records in splits.items()}

all_split_paths = []
for records in splits.values():
    all_split_paths.extend(str(path.resolve()) for path, _ in records)
assert len(all_split_paths) == len(set(all_split_paths)), 'Data leakage detected: duplicated original path across splits.'

print('Split sizes:', split_sizes)
print('Split label counts:', split_counts)

split_names = list(splits.keys())
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

class_values = [class_counts[class_name] for class_name in CLASS_NAMES]
bars = axes[0].bar(CLASS_NAMES, class_values, color=['#2f80ed', '#27ae60'])
axes[0].set_title('Distribusi Class Dataset Asli')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Jumlah gambar')
axes[0].bar_label(bars)

bottom = np.zeros(len(split_names))
for class_name, color in zip(CLASS_NAMES, ['#2f80ed', '#27ae60']):
    values = [split_counts[split_name][class_name] for split_name in split_names]
    axes[1].bar(split_names, values, bottom=bottom, label=class_name, color=color)
    bottom += np.array(values)
axes[1].set_title('Distribusi Label per Split')
axes[1].set_xlabel('Split')
axes[1].set_ylabel('Jumlah gambar')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_class_split_distribution.png', dpi=160)
plt.show()

eda_summary['split_ratio'] = {'train': TRAIN_RATIO, 'validation': VAL_RATIO, 'test': TEST_RATIO}
eda_summary['split_sizes'] = split_sizes
eda_summary['split_label_counts'] = split_counts
eda_summary['figures']['class_split_distribution'] = str(OUTPUT_DIR / 'eda_class_split_distribution.png')
